# Перебор гиперпараметров

Цель: найти оптимальные параметры для лучших моделей (LightGBM, XGBoost, RandomForest) с помощью RandomizedSearchCV.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb
import xgboost as xgb
from src.preprocessing.clean import clean_all
from src.preprocessing.features import build_features
from src.models.evaluate import split_data, regression_metrics, print_metrics
from src.models.train import build_feature_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
df = build_features(clean_all())
X, y, cat_indices, col_names = build_feature_matrix(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, random_state=RANDOM_SEED)

num_cols = [c for i, c in enumerate(col_names) if i not in cat_indices]
X_train_num = X_train[num_cols].astype(float)
X_val_num = X_val[num_cols].astype(float)
X_test_num = X_test[num_cols].astype(float)

print(f'Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')
print(f'Числовых признаков: {len(num_cols)}, всего: {len(col_names)}')

tuning_results = []

## RandomForest - RandomizedSearchCV

**Гипотеза:** подбор числа деревьев и глубины улучшит MAPE RandomForest без риска переобучения.

In [ ]:
rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)),
])

rf_param_dist = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', 0.5, 0.7],
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_percentage_error',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=1,
)
rf_search.fit(X_train_num, y_train)

print('Лучшие параметры RF:', rf_search.best_params_)
m_rf = regression_metrics(y_val, rf_search.predict(X_val_num))
print_metrics('RandomForest tuned (val)', m_rf)
tuning_results.append({'model': 'RandomForest (tuned)', **m_rf, 'best_params': str(rf_search.best_params_)})

## XGBoost - RandomizedSearchCV

**Гипотеза:** подбор learning_rate и регуляризации снизит ошибку XGBoost.

In [ ]:
xgb_model = xgb.XGBRegressor(
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=0,
)

xgb_param_dist = {
    'n_estimators': [200, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1, 2, 5],
}

X_train_xgb = X_train_num.fillna(-999)
X_val_xgb = X_val_num.fillna(-999)

xgb_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=xgb_param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_percentage_error',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=1,
)
xgb_search.fit(X_train_xgb, y_train)

print('Лучшие параметры XGB:', xgb_search.best_params_)
m_xgb = regression_metrics(y_val, xgb_search.predict(X_val_xgb))
print_metrics('XGBoost tuned (val)', m_xgb)
tuning_results.append({'model': 'XGBoost (tuned)', **m_xgb, 'best_params': str(xgb_search.best_params_)})

## LightGBM - RandomizedSearchCV

**Гипотеза:** LightGBM с категориальными признаками и подобранными num_leaves/learning_rate даёт лучшее качество.

In [ ]:
lgb_model = lgb.LGBMRegressor(random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)

lgb_param_dist = {
    'n_estimators': [300, 500, 800, 1000],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'num_leaves': [31, 63, 127, 255],
    'max_depth': [-1, 5, 7, 10],
    'min_child_samples': [10, 20, 50],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 1.0],
}

lgb_search = RandomizedSearchCV(
    lgb_model,
    param_distributions=lgb_param_dist,
    n_iter=30,
    cv=3,
    scoring='neg_mean_absolute_percentage_error',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=1,
)
lgb_search.fit(X_train, y_train)

print('Лучшие параметры LGB:', lgb_search.best_params_)
m_lgb = regression_metrics(y_val, lgb_search.predict(X_val))
print_metrics('LightGBM tuned (val)', m_lgb)
tuning_results.append({'model': 'LightGBM (tuned)', **m_lgb, 'best_params': str(lgb_search.best_params_)})

## Сравнение результатов настройки

In [ ]:
tuning_df = pd.DataFrame(tuning_results)[['model', 'mae', 'rmse', 'mape']].sort_values('mape')
display(tuning_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(tuning_df['model'], tuning_df['mape'])
ax.set_xlabel('MAPE (%)')
ax.set_title('Сравнение настроенных моделей по MAPE (меньше = лучше)')
ax.invert_yaxis()
plt.tight_layout()

## Финальная оценка лучшей настроенной модели на тесте

In [ ]:
best_name = tuning_df.iloc[0]['model']
print(f'Лучшая настроенная модель: {best_name}')

m_test = regression_metrics(y_test, lgb_search.predict(X_test))
print_metrics('LightGBM tuned - TEST', m_test)

tuning_df.to_csv(ROOT / 'data/processed/tuning_results.csv', index=False)